In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

zip_path = "/content/drive/MyDrive/VeritasAI/dataset/real-vs-fake.zip"

print("Exists:", os.path.exists(zip_path))

Exists: True


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/VeritasAI/dataset/real-vs-fake.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")

Extraction complete.


In [ ]:
import os

for root, dirs, files in os.walk("/content/dataset"):
    print(root)
    print("Folders:", dirs[:10])
    break

/content/dataset
Folders: ['real-vs-fake']


In [ ]:
for root, dirs, files in os.walk("/content/dataset"):
    if "train" in dirs:
        print("Found train folder at:")
        print(root)

Found train folder at:
/content/dataset/real-vs-fake


In [ ]:
import os

folders = [
    "/content/dataset/real-vs-fake/train/real",
    "/content/dataset/real-vs-fake/train/fake",
    "/content/dataset/real-vs-fake/valid/real",
    "/content/dataset/real-vs-fake/valid/fake",
    "/content/dataset/real-vs-fake/test/real",
    "/content/dataset/real-vs-fake/test/fake",
]

for folder in folders:
    print(folder)
    print("Images:", len(os.listdir(folder)))
    print()

/content/dataset/real-vs-fake/train/real
Images: 50000

/content/dataset/real-vs-fake/train/fake
Images: 50000

/content/dataset/real-vs-fake/valid/real
Images: 10000

/content/dataset/real-vs-fake/valid/fake
Images: 10000

/content/dataset/real-vs-fake/test/real
Images: 10000

/content/dataset/real-vs-fake/test/fake
Images: 10000



In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
TRAIN_DIR = "/content/dataset/real-vs-fake/train"
VALID_DIR = "/content/dataset/real-vs-fake/valid"
TEST_DIR  = "/content/dataset/real-vs-fake/test"

In [ ]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
print(train_data.class_indices)

{'fake': 0, 'real': 1}


In [ ]:
print(tf.__version__)

2.20.0


In [ ]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!nvidia-smi

Sat Jun  6 14:22:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
train_datagen.flow_from_directory(...)

TypeError: listdir: path should be string, bytes, os.PathLike, integer or None, not ellipsis

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
print("Train:", train_data.samples)
print("Valid:", valid_data.samples)
print("Test:", test_data.samples)

print("Classes:", train_data.class_indices)

Train: 100000
Valid: 20000
Test: 20000
Classes: {'fake': 0, 'real': 1}


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!nvidia-smi

Sat Jun  6 14:24:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from tensorflow.keras.applications import Xception

base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(299, 299, 3)
)

base_model.trainable = False

print("Xception loaded successfully")

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Xception loaded successfully


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 10, 10, 2048)   │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,123,881 (80.58 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [ ]:
base_model.trainable = False

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully")

Model compiled successfully


In [ ]:
import os

os.makedirs(
    "/content/drive/MyDrive/VeritasAI/models",
    exist_ok=True
)

print("Models folder ready")

Models folder ready


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=1,
    callbacks=[early_stop, checkpoint]
)

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 827ms/step - accuracy: 0.7260 - loss: 0.5391
Epoch 1: val_loss improved from None to 0.43316, saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2795s 881ms/step - accuracy: 0.7467 - loss: 0.5119 - val_accuracy: 0.8012 - val_loss: 0.4332


In [ ]:
model.evaluate(test_data)

625/625 ━━━━━━━━━━━━━━━━━━━━ 156s 250ms/step - accuracy: 0.8009 - loss: 0.4368


[0.436842679977417, 0.8008999824523926]

In [ ]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
model

NameError: name 'model' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input

In [ ]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [ ]:
TRAIN_DIR = "/content/dataset/real-vs-fake/train"
VALID_DIR = "/content/dataset/real-vs-fake/valid"
TEST_DIR  = "/content/dataset/real-vs-fake/test"

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/real-vs-fake/train'

In [ ]:
import os

print(os.path.exists("/content/dataset/real-vs-fake"))

False


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/VeritasAI/dataset/real-vs-fake.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete")

Extraction complete


In [ ]:
import os

print(os.path.exists("/content/dataset/real-vs-fake/train"))

True


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input

IMG_SIZE = (299, 299)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

TRAIN_DIR = "/content/dataset/real-vs-fake/train"
VALID_DIR = "/content/dataset/real-vs-fake/valid"
TEST_DIR  = "/content/dataset/real-vs-fake/test"

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
model.evaluate(test_data)

NameError: name 'model' is not defined

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras"
)

print("Model loaded successfully")

Model loaded successfully


In [ ]:
model.evaluate(test_data)

625/625 ━━━━━━━━━━━━━━━━━━━━ 220s 289ms/step - accuracy: 0.8009 - loss: 0.4368


[0.436842679977417, 0.8008999824523926]

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - accuracy: 0.7741 - loss: 0.4752
Epoch 1: val_loss improved from None to 0.38865, saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras

Epoch 1: finished saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2830s 900ms/step - accuracy: 0.7779 - loss: 0.4683 - val_accuracy: 0.8267 - val_loss: 0.3886
Epoch 2/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 814ms/step - accuracy: 0.7808 - loss: 0.4594
Epoch 2: val_loss improved from 0.38865 to 0.37256, saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras

Epoch 2: finished saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2741s 873ms/step - accuracy: 0.7843 - loss: 0.4557 - val_accuracy: 0.8360 - val_loss: 0.3726
Epoch 3/10
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 840ms/step - accuracy: 0.7898 - loss: 0.4457
Epoch 3: val_lo

KeyboardInterrupt: 

In [ ]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras"
)

model.evaluate(test_data)

625/625 ━━━━━━━━━━━━━━━━━━━━ 198s 295ms/step - accuracy: 0.8403 - loss: 0.3721


[0.3721361458301544, 0.8403000235557556]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/VeritasAI/dataset/real-vs-fake.zip"
extract_path = "/content/dataset"

if not os.path.exists("/content/dataset/real-vs-fake"):
    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("Dataset extracted successfully!")

else:
    print("Dataset already exists.")

Extracting dataset...
Dataset extracted successfully!


In [ ]:
import os

print("Train Exists:",
      os.path.exists("/content/dataset/real-vs-fake/train"))

print("Valid Exists:",
      os.path.exists("/content/dataset/real-vs-fake/valid"))

print("Test Exists:",
      os.path.exists("/content/dataset/real-vs-fake/test"))

Train Exists: True
Valid Exists: True
Test Exists: True


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input

IMG_SIZE = (299, 299)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

TRAIN_DIR = "/content/dataset/real-vs-fake/train"
VALID_DIR = "/content/dataset/real-vs-fake/valid"
TEST_DIR  = "/content/dataset/real-vs-fake/test"

In [ ]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [ ]:
print("Train:", train_data.samples)
print("Valid:", valid_data.samples)
print("Test :", test_data.samples)

print("Classes:", train_data.class_indices)

Train: 100000
Valid: 20000
Test : 20000
Classes: {'fake': 0, 'real': 1}


In [ ]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras"
)

print("Veritas AI model loaded successfully!")

Veritas AI model loaded successfully!


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 10, 10, 2048)   │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,648,685 (82.58 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 20,861,480 (79.58 MB)

 Optimizer params: 524,804 (2.00 MB)

In [ ]:
model.evaluate(test_data)

625/625 ━━━━━━━━━━━━━━━━━━━━ 193s 250ms/step - accuracy: 0.8403 - loss: 0.3721


[0.3721361458301544, 0.8403000235557556]

In [ ]:

base_model = model.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 10, 10, 2048)   │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,648,685 (82.58 MB)

 Trainable params: 9,202,753 (35.11 MB)

 Non-trainable params: 11,921,128 (45.48 MB)

 Optimizer params: 524,804 (2.00 MB)

In [ ]:
print(sum([layer.trainable for layer in base_model.layers]))
print(len(base_model.layers))

30
132


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [ ]:
model.evaluate(test_data)

NameError: name 'model' is not defined

In [ ]:
import os

print(os.path.exists("/content/dataset/real-vs-fake/train"))

False


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile

zip_path = "/content/drive/MyDrive/VeritasAI/dataset/real-vs-fake.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted")

Dataset extracted


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.xception import preprocess_input

IMG_SIZE = (299, 299)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

TRAIN_DIR = "/content/dataset/real-vs-fake/train"
VALID_DIR = "/content/dataset/real-vs-fake/valid"
TEST_DIR = "/content/dataset/real-vs-fake/test"

In [5]:
train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [6]:
from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v1.keras"
)

print("Model loaded")

Model loaded


In [7]:
base_model = model.layers[0]

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [8]:
print(sum(layer.trainable for layer in base_model.layers))
print(len(base_model.layers))

30
132


In [9]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [11]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=2,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/2
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 822ms/step - accuracy: 0.7768 - loss: 0.5709
Epoch 1: val_loss improved from None to 0.15822, saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras

Epoch 1: finished saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2817s 884ms/step - accuracy: 0.8489 - loss: 0.3611 - val_accuracy: 0.9388 - val_loss: 0.1582
Epoch 2/2
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 832ms/step - accuracy: 0.9293 - loss: 0.1806
Epoch 2: val_loss improved from 0.15822 to 0.09916, saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras

Epoch 2: finished saving model to /content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2782s 890ms/step - accuracy: 0.9354 - loss: 0.1648 - val_accuracy: 0.9632 - val_loss: 0.0992


In [12]:

from tensorflow.keras.models import load_model

model = load_model(
    "/content/drive/MyDrive/VeritasAI/models/veritas_xception_v2_finetuned.keras"
)

In [13]:
model.evaluate(test_data)

625/625 ━━━━━━━━━━━━━━━━━━━━ 193s 287ms/step - accuracy: 0.9606 - loss: 0.1036


[0.10360889881849289, 0.9606000185012817]